# Perbaikan Pipe 01 & 02 — MDA Proton Transfer

Implementasi perbaikan berdasarkan analisis riset:
- **Pipe 01** (Ground State): basis `6-31G*`, CAS(8e,6o), reps=3, SLSQP, multi-start
- **Pipe 02** (PES Scan): grid adaptif dekat TS, checkpoint, CASSCF benchmark, C₂ᵥ simetri

Menggunakan **Qiskit 2.3.0** + qiskit-nature + PySCF.

In [ ]:
# Install semua dependensi yang dibutuhkan
# Jalankan sekali, restart kernel setelahnya
!pip install "qiskit==2.3.0" qiskit-algorithms "qiskit-nature[pyscf]" qiskit-aer pyscf -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json, os, time

import qiskit
from qiskit.primitives import StatevectorEstimator
from qiskit.circuit.library import EfficientSU2
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SLSQP, L_BFGS_B
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from pyscf import gto, scf, mcscf
import scipy.sparse.linalg as spla

print(f"Qiskit: {qiskit.__version__}")

## Pipe 01 — Ground State MDA (Perbaikan)

| Aspek | Sebelum | Sesudah |
|---|---|---|
| Basis set | STO-3G | **6-31G\*** |
| Active space | CAS(6e,4o) → 8 qubit | **CAS(8e,6o) → 10 qubit** |
| Reps ansatz | 2 | **3** |
| Optimizer | COBYLA | **SLSQP** |
| Orbital reorder | — | **sort_mo_by_energy()** |
| Multi-start | — | **3 titik awal** |

In [ ]:
# ===== Parameter Global (semua perbaikan terpusat di sini) =====
BASIS    = '6-31g*'   # perbaikan A.1: dari STO-3G
N_ELEC   = 8          # perbaikan A.3: dari 6 ke 8 elektron aktif
N_ORB    = 6          # perbaikan A.3: dari 4 ke 6 orbital aktif
REPS     = 3          # perbaikan B.1: dari 2 ke 3
N_START  = 3          # perbaikan B.3: multi-start
OPT_NAME = 'SLSQP'   # perbaikan B.2: ganti COBYLA

# ===== Koordinat MDA enol form (C2v, Angstrom) =====
# Backbone C3 dengan dua O di ujung; Ht = transferring H
O1_POS = np.array([0.000,  1.295,  0.900])
O2_POS = np.array([0.000, -1.295,  0.900])

BASE_ATOMS = [
    ('O',  O1_POS),
    ('C',  np.array([0.000,  1.220, -0.280])),
    ('C',  np.array([0.000,  0.000, -0.850])),
    ('C',  np.array([0.000, -1.220, -0.280])),
    ('O',  O2_POS),
    ('H',  np.array([0.000,  2.280, -0.700])),
    ('H',  np.array([0.000,  0.000, -1.940])),
    ('H',  np.array([0.000, -2.280, -0.700])),
]

def get_ht_position(q):
    """Posisi H transfer pada koordinat q.
    q=0: midpoint TS, q<0: dekat O1, q>0: dekat O2."""
    midpoint  = 0.5 * (O1_POS + O2_POS)
    direction = O2_POS - O1_POS  # vektor O1→O2
    return midpoint + q * direction

def build_atom_str(q=0.0):
    """Bangun atom string PySCFDriver-compatible pada koordinat q."""
    atoms = BASE_ATOMS + [('H', get_ht_position(q))]
    return "; ".join(
        f"{sym} {r[0]:.4f} {r[1]:.4f} {r[2]:.4f}" for sym, r in atoms
    )

def build_mda_mol(q=0.0, sym=True):
    """Buat PySCF Mole untuk MDA di koordinat transfer proton q."""
    mol = gto.Mole()
    mol.atom     = build_atom_str(q)
    mol.basis    = BASIS   # 6-31g* (perbaikan A.1)
    mol.symmetry = sym     # perbaikan C.2: enforce C2v
    mol.charge   = 0
    mol.spin     = 0
    mol.verbose  = 0
    mol.build()
    return mol

# Verifikasi simetri C2v di TS (perbaikan C.2)
mol_ts = build_mda_mol(q=0.0)
print(f"Simetri (q=0): {mol_ts.groupname}")
print(f"Jumlah atom: {mol_ts.natm}")

In [ ]:
def run_casscf(mol):
    """
    HF → CASSCF dengan orbital reordering (perbaikan A.2 & A.3).
    mcscf.addons.sort_mo_by_energy() memastikan lone pair O dan pi/sigma*(O-H) masuk active space.
    Mengembalikan (mf, mc) untuk benchmark klasik (perbaikan D.3).
    """
    from pyscf.mcscf import addons  # import fungsi sort orbital eksplisit

    # Langkah 1: Restricted Hartree-Fock sebagai referensi
    mf = scf.RHF(mol)
    mf.kernel()

    # Langkah 2: Setup CASSCF dengan CAS(8e,6o)
    mc = mcscf.CASSCF(mf, N_ORB, N_ELEC)

    # Orbital reordering (perbaikan A.2)
    # sort_mo_by_energy bukan method objek, melainkan fungsi di mcscf.addons
    # Hasilnya (mo_sorted) langsung dikirim ke mc.kernel() sebagai initial MO
    mo_sorted = addons.sort_mo_by_energy(mc, mf.mo_coeff)

    # Langkah 3: Jalankan CASSCF dengan MO yang sudah diurutkan
    mc.kernel(mo_sorted)
    return mf, mc

# Test CASSCF pada TS
print('Menjalankan CASSCF pada q=0 (TS)...')
_, mc_test = run_casscf(mol_ts)
print(f"E_CASSCF(TS) = {mc_test.e_tot:.6f} Ha")
print(f"Konvergen   : {mc_test.converged}")

In [ ]:
def get_qubit_hamiltonian(q=0.0):
    """
    Bangun qubit Hamiltonian MDA pada koordinat q.
    CAS(8e,6o) + Parity mapper + 2-qubit reduction → 10 qubit.
    """
    # Driver PySCF dengan basis 6-31g* (perbaikan A.1)
    driver = PySCFDriver(
        atom=build_atom_str(q),
        basis=BASIS,
        charge=0,
        spin=0,
    )
    problem = driver.run()

    # Active space transformer CAS(8e,6o) (perbaikan A.3)
    transformer = ActiveSpaceTransformer(
        num_electrons=N_ELEC,
        num_spatial_orbitals=N_ORB,
    )
    problem = transformer.transform(problem)

    # Parity mapper + 2-qubit reduction → 10 qubit
    mapper   = ParityMapper(num_particles=problem.num_particles)
    qubit_op = mapper.map(problem.second_q_ops()[0])

    return qubit_op, qubit_op.num_qubits

# Build Hamiltonian di TS dan cek ukurannya
print('Membangun qubit Hamiltonian pada q=0...')
H_op, n_qb = get_qubit_hamiltonian(q=0.0)
print(f"Jumlah qubit     : {n_qb}")
print(f"Jumlah Pauli term: {len(H_op)}")

In [ ]:
def run_multistart_vqe(qubit_op, n_qubits, warm_params=None):
    """
    VQE dengan multi-start untuk menghindari local minimum (perbaikan B.3).
    Ansatz EfficientSU2 reps=3 (perbaikan B.1).
    Optimizer SLSQP/L-BFGS-B gradient-based (perbaikan B.2).

    Parameters
    ----------
    qubit_op    : SparsePauliOp — qubit Hamiltonian
    n_qubits    : int           — jumlah qubit
    warm_params : ndarray|None  — parameter dari titik grid sebelumnya

    Returns
    -------
    (best_result, best_energy, optimal_params)
    """
    # Estimator statevector (exact sim)
    estimator = StatevectorEstimator()

    # Ansatz EfficientSU2 dengan reps ditingkatkan (perbaikan B.1)
    ansatz = EfficientSU2(
        num_qubits=n_qubits,
        reps=REPS,             # reps=3 (dari reps=2)
        entanglement='linear',
    )
    n_params = ansatz.num_parameters

    # Optimizer gradient-based (perbaikan B.2)
    optimizer = SLSQP(maxiter=500) if OPT_NAME == 'SLSQP' else L_BFGS_B(maxiter=500)

    best_energy = np.inf
    best_result = None
    best_params = None

    # Kumpulkan titik awal: warm start (jika ada) + N_START random
    starts = []
    if warm_params is not None:
        starts.append(('warm', warm_params))   # parameter dari grid sebelumnya
    for i in range(N_START):
        starts.append((f'rand{i}', np.random.uniform(-np.pi, np.pi, n_params)))

    # Multi-start loop (perbaikan B.3)
    for label, init_p in starts:
        vqe = VQE(
            estimator=estimator,
            ansatz=ansatz,
            optimizer=optimizer,
            initial_point=init_p,
        )
        result = vqe.compute_minimum_eigenvalue(operator=qubit_op)
        e = result.eigenvalue.real
        print(f"  [{label}]: E = {e:.6f} Ha")

        if e < best_energy:
            best_energy = e
            best_result = result
            best_params = result.optimal_point

    return best_result, best_energy, best_params

In [ ]:
print("=" * 55)
print("  Pipe 01: Ground State MDA — q=0 (Transition State)")
print("=" * 55)

# Qubit Hamiltonian di TS
H_op, n_qb = get_qubit_hamiltonian(q=0.0)
print(f"\nQubit Hamiltonian: {n_qb} qubit, {len(H_op)} Pauli terms")

# [1] Exact diagonalization sebagai referensi
print('\n[1] Exact diagonalization...')
H_mat = H_op.to_matrix(sparse=True)
evals, _ = spla.eigsh(H_mat, k=1, which='SA')
e_exact = evals[0]
print(f"    E_exact = {e_exact:.6f} Ha")

# [2] Multi-start VQE
print(f"\n[2] VQE multi-start ({N_START} starts, reps={REPS}, {OPT_NAME})...")
result_p01, e_vqe_p01, warm_p01 = run_multistart_vqe(H_op, n_qb)
print(f"\n    E_VQE_best = {e_vqe_p01:.6f} Ha")
print(f"    Selisih    = {(e_vqe_p01 - e_exact)*1000:.4f} mHa")

# [3] CASSCF klasik benchmark (perbaikan D.3)
print('\n[3] CASSCF klasik (benchmark)...')
_, mc_p01 = run_casscf(build_mda_mol(q=0.0))
print(f"    E_CASSCF = {mc_p01.e_tot:.6f} Ha")

## Pipe 02 — Konstruksi PES MDA (Perbaikan)

| Aspek | Sebelum | Sesudah |
|---|---|---|
| Grid | 13 titik uniform | **Grid adaptif (rapat di TS)** |
| Checkpoint | — | **Simpan per titik (.json)** |
| Benchmark | — | **CASSCF klasik di tiap titik** |
| Warm start | Parameter random | **Transfer params dari titik sebelumnya** |

In [ ]:
# Grid adaptif (perbaikan C.3)
# Rapat di sekitar TS (|q| < 0.2 Ang), jarang di sumur potensial

q_dense = np.linspace(-0.20, 0.20, 7)         # 7 titik rapat dekat TS
q_left  = np.array([-0.50, -0.40, -0.30])     # 3 titik sumur kiri
q_right = np.array([ 0.30,  0.40,  0.50])     # 3 titik sumur kanan

# Gabung dan sort (hapus duplikat)
Q_GRID = np.sort(np.unique(np.concatenate([q_left, q_dense, q_right])))
print(f"Adaptive grid: {len(Q_GRID)} titik")
print(f"q = {Q_GRID.round(3)}")

# Visualisasi distribusi grid
fig, ax = plt.subplots(figsize=(9, 1.8))
ax.scatter(Q_GRID, np.zeros_like(Q_GRID), marker='|', s=300,
           c='steelblue', linewidths=2.5)
ax.axvspan(-0.2, 0.2, alpha=0.15, color='tomato', label='Dense near TS')
ax.set_xlabel('q (Å)'); ax.set_yticks([])
ax.set_title('Adaptive Grid — Distribusi Titik Kalkulasi'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# PES scan dengan checkpoint (D.1) dan CASSCF benchmark (D.3)
# ===========================================================
CKPT_FILE = 'pes_mda_ckpt.json'

def load_ckpt():
    """Load checkpoint jika file sudah ada."""
    if os.path.exists(CKPT_FILE):
        with open(CKPT_FILE) as f:
            return json.load(f)
    return {}

def save_ckpt(data):
    """Simpan progress ke file setelah setiap titik berhasil (perbaikan D.1)."""
    with open(CKPT_FILE, 'w') as f:
        json.dump(data, f, indent=2)

# Muat checkpoint dan inisialisasi warm start dari Pipe 01
pes        = load_ckpt()
warm_p     = warm_p01   # parameter awal dari single-point Pipe 01
print(f"Checkpoint: {len(pes)} titik sudah selesai\n")

for q in Q_GRID:
    key = f'{q:.4f}'

    # Skip jika sudah ada di checkpoint — lanjut dari titik terakhir (perbaikan D.1)
    if key in pes:
        print(f"q={q:+.3f}: skip (checkpoint)")
        # Ambil params dari checkpoint untuk warm start ke depan
        if pes[key].get('params'):
            warm_p = np.array(pes[key]['params'])
        continue

    print(f"q={q:+.3f}: menghitung...", end="", flush=True)
    t0 = time.time()

    # --- CASSCF klasik: benchmark di setiap titik (perbaikan D.3) ---
    mol_q        = build_mda_mol(q)
    _, mc_q      = run_casscf(mol_q)
    e_casscf_q   = mc_q.e_tot

    # --- Qubit Hamiltonian + exact diagonalization ---
    H_op_q, n_qb_q = get_qubit_hamiltonian(q)
    H_mat_q = H_op_q.to_matrix(sparse=True)
    ev_q, _ = spla.eigsh(H_mat_q, k=1, which='SA')
    e_exact_q   = ev_q[0]

    # --- VQE dengan warm start dari titik sebelumnya (perbaikan D.1 transfer params) ---
    _, e_vqe_q, warm_p = run_multistart_vqe(H_op_q, n_qb_q, warm_params=warm_p)

    # Simpan ke checkpoint
    pes[key] = {
        'q':      float(q),
        'vqe':    float(e_vqe_q),
        'casscf': float(e_casscf_q),
        'exact':  float(e_exact_q),
        'params': warm_p.tolist() if warm_p is not None else None,
    }
    save_ckpt(pes)

    dt = time.time() - t0
    print(f"  E_vqe={e_vqe_q:.5f}, E_casscf={e_casscf_q:.5f}, E_exact={e_exact_q:.5f}  ({dt:.1f}s)")

print(f"\nSelesai! {len(pes)} titik total.")

In [ ]:
# Susun data dari dict checkpoint
keys        = sorted(pes.keys(), key=float)
qs_pl       = np.array([pes[k]['q']      for k in keys])
e_vqe_pl    = np.array([pes[k]['vqe']    for k in keys])
e_casscf_pl = np.array([pes[k]['casscf'] for k in keys])
e_exact_pl  = np.array([pes[k]['exact']  for k in keys])

# Konversi ke energi relatif (kcal/mol) terhadap minimum masing-masing
ha2kcal = 627.509
rel = lambda e: (e - e.min()) * ha2kcal

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: kurva PES
ax1.plot(qs_pl, rel(e_exact_pl),  'k-o',  lw=2,   ms=5, label='Exact Diag')
ax1.plot(qs_pl, rel(e_casscf_pl), 'b--s', lw=1.5, ms=5, label='CASSCF (klasik)')
ax1.plot(qs_pl, rel(e_vqe_pl),    'r-^',  lw=1.5, ms=5, label='VQE')
ax1.axvline(0, color='gray', ls=':', alpha=0.5, label='TS (q=0)')
ax1.set_xlabel('q (Å)'); ax1.set_ylabel('ΔE (kcal/mol)')
ax1.set_title('PES MDA — Proton Transfer')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

# Plot 2: error VQE dan CASSCF terhadap Exact (mHa)
err_vqe    = (e_vqe_pl    - e_exact_pl) * 1000
err_casscf = (e_casscf_pl - e_exact_pl) * 1000
w = 0.012
ax2.bar(qs_pl - w, err_vqe,    2*w, label='VQE - Exact',    color='salmon',    alpha=0.85)
ax2.bar(qs_pl + w, err_casscf, 2*w, label='CASSCF - Exact', color='steelblue', alpha=0.85)
ax2.axhline(0, color='k', lw=0.8)
ax2.set_xlabel('q (Å)'); ax2.set_ylabel('Error (mHa)')
ax2.set_title('Error vs Exact Diag')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Ringkasan barrier height
barrier = lambda e: (e.max() - e.min()) * ha2kcal
print(f"\nBarrier height (TS → minimum):")
print(f"  Exact:  {barrier(e_exact_pl):.2f} kcal/mol")
print(f"  VQE:    {barrier(e_vqe_pl):.2f} kcal/mol  (Δ={barrier(e_vqe_pl)-barrier(e_exact_pl):+.2f})")
print(f"  CASSCF: {barrier(e_casscf_pl):.2f} kcal/mol  (Δ={barrier(e_casscf_pl)-barrier(e_exact_pl):+.2f})")